In [1]:
# ==========================================
# Fleet Data Analysis using NLP + K-Means
# ==========================================

# Import libraries

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from sklearn.metrics import silhouette_score


# ==========================================
# 1. Load Fleet Dataset
# ==========================================

df = pd.read_csv("Fleet Data.csv")

# Display first rows

df.head()


# Dataset information

df.info()


# Check missing values

df.isnull().sum()



# ==========================================
# 2. Data Cleaning
# ==========================================

# Remove duplicate rows

df = df.drop_duplicates()


# Fill missing values

df = df.fillna("Unknown")


# View columns

print(df.columns)



# ==========================================
# 3. Numerical Feature Preparation
# ==========================================

# Select numerical columns

numeric_features = df.select_dtypes(
    include=['int64','float64']
)


numeric_features.head()



# Scale numerical data

scaler = StandardScaler()

scaled_numeric = scaler.fit_transform(
    numeric_features
)



# ==========================================
# 4. NLP Processing
# ==========================================

# Choose text column
# Change this name to your actual text column

text_column = "Description"


if text_column in df.columns:

    text_data = df[text_column].astype(str)

    # TF-IDF conversion

    tfidf = TfidfVectorizer(
        stop_words="english",
        max_features=500
    )


    text_features = tfidf.fit_transform(
        text_data
    )


    text_features = text_features.toarray()


else:

    print("No text column found")

    text_features = np.zeros(
        (len(df),1)
    )



# ==========================================
# 5. Combine NLP + Numerical Features
# ==========================================

combined_features = np.hstack(
    [
        scaled_numeric,
        text_features
    ]
)


combined_features.shape



# ==========================================
# 6. Find Optimal Number of Clusters
# ==========================================

scores = []

K_range = range(2,10)


for k in K_range:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(
        combined_features
    )

    score = silhouette_score(
        combined_features,
        labels
    )

    scores.append(score)



plt.figure(figsize=(8,5))

plt.plot(
    list(K_range),
    scores,
    marker="o"
)

plt.xlabel(
    "Number of Clusters"
)

plt.ylabel(
    "Silhouette Score"
)

plt.title(
    "Choosing Optimal K"
)

plt.show()



# ==========================================
# 7. Apply K-Means
# ==========================================

k = 4   # Change after evaluating graph


kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)


df["Cluster"] = kmeans.fit_predict(
    combined_features
)



# View clusters

df.head()



# ==========================================
# 8. Cluster Analysis
# ==========================================

cluster_summary = df.groupby(
    "Cluster"
).size()


cluster_summary



# ==========================================
# 9. PCA Visualisation
# ==========================================

pca = PCA(
    n_components=2
)


pca_features = pca.fit_transform(
    combined_features
)


plt.figure(figsize=(8,6))


plt.scatter(
    pca_features[:,0],
    pca_features[:,1],
    c=df["Cluster"]
)


plt.xlabel("PCA 1")

plt.ylabel("PCA 2")

plt.title(
    "Fleet Data K-Means Clusters"
)

plt.show()



# ==========================================
# 10. Save Results
# ==========================================

df.to_csv(
    "Fleet_Data_Clustered.csv",
    index=False
)

print(
    "Clustering completed successfully"
)

<class 'pandas.DataFrame'>
RangeIndex: 1583 entries, 0 to 1582
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Parent Airline        1583 non-null   str    
 1   Airline               1583 non-null   str    
 2   Aircraft Type         1583 non-null   str    
 3   Current               859 non-null    float64
 4   Future                188 non-null    float64
 5   Historic              1113 non-null   float64
 6   Total                 1484 non-null   float64
 7   Orders                348 non-null    float64
 8   Unit Cost             1548 non-null   str    
 9   Total Cost (Current)  1556 non-null   str    
 10  Average Age           820 non-null    float64
dtypes: float64(6), str(5)
memory usage: 209.8 KB
Index(['Parent Airline', 'Airline', 'Aircraft Type', 'Current', 'Future',
       'Historic', 'Total', 'Orders', 'Unit Cost', 'Total Cost (Current)',
       'Average Age'],
      dtype='str')


ValueError: at least one array or dtype is required